## 第 1 课：第一个 Kernel 与 1-D Grid

题目：[Triton: Write a Constant to Every Element](https://www.deep-ml.com/problems/967?from=Triton%20Essentials)（ID 967）

计算目标：

In [ ]:
output[i] = value   (0 <= i < n)

output 是长度为 `n` 的 1-D `float32` Tensor，所有元素都等于 `value`。

例如：

In [ ]:
n = 5, value = 3.0

output = [3.0, 3.0, 3.0, 3.0, 3.0]

这个 kernel **没有输入 Tensor，只有一个输出**。它存在的全部意义就是教你 launch pattern（启动模式）和带 mask 的 `tl.store`，没有任何输入 load 和算术来干扰你。

### 1. 1-D Grid 与 SPMD

Triton 的程序是 SPMD（Single Program, Multiple Data）：同一个 kernel 被复制成很多个 program 并行执行，每个 program 只负责一部分数据。program 怎么知道"我是谁"？靠 `tl.program_id`：

In [ ]:
pid = tl.program_id(axis=0)  # 我是第几个 program

1-D 时 axis 只有 0。总共启动多少个 program 由 grid 决定：

In [ ]:
grid = (triton.cdiv(n, BLOCK_SIZE),)

每个 program 负责连续 `BLOCK_SIZE` 个元素：

In [ ]:
pid=0 → 元素 0          ~ BLOCK_SIZE-1
pid=1 → 元素 BLOCK_SIZE ~ 2*BLOCK_SIZE-1
pid=2 → 元素 2*BLOCK_SIZE ~ 3*BLOCK_SIZE-1
...

### 2. 当前 program 负责的元素下标

In [ ]:
offsets = (
    pid * BLOCK_SIZE
    + tl.arange(0, BLOCK_SIZE)
)

`tl.arange(0, BLOCK_SIZE)` 生成 `0, 1, ..., BLOCK_SIZE-1`，加上 `pid * BLOCK_SIZE` 平移，就得到这个 program 负责的区间。

### 3. Mask：防止越界

如果 `n` 不是 `BLOCK_SIZE` 的整数倍，最后一个 program 会"超出"数组末尾：

In [ ]:
mask = offsets < n

`tl.store(output_ptr + offsets, value, mask=mask)` 只写 mask 为 True 的 lane，越界的 lane 什么都不做。

两个注意点：

- 没有输入，所以不需要 `tl.load`，`value` 直接就是写出去的值。
- `value` 是**运行时标量参数**，不是 `tl.constexpr` —— 我们希望换不同的 value 时不用重新编译。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def fill_kernel(
    output_ptr,
    n,
    value,
    BLOCK_SIZE: tl.constexpr,
):
    # TODO 1：取得当前 program 的 ID

    # TODO 2：生成当前块的元素下标 offsets

    # TODO 3：生成 mask（防止越界写）

    # TODO 4：把 value 写入 output
    # 注意：没有输入，不需要 tl.load
    pass


def fill(n: int, value: float) -> torch.Tensor:
    BLOCK_SIZE = 1024

    # TODO 5：分配 output（torch.empty，形状 (n,)，dtype=torch.float32）

    # TODO 6：创建一维 grid（triton.cdiv）

    # TODO 7：启动 kernel

    # TODO 8：返回 output
    pass

同时回答：

1. `n = 10000, BLOCK_SIZE = 1024` 时，grid 是多少？最后一个 program（pid=9）实际写入多少个有效元素？
2. 为什么 `value` 是普通运行时参数而不是 `tl.constexpr`？如果改成 constexpr 会有什么代价？
3. 这个 kernel 没有输入、只有输出，为什么仍然需要 mask？

把代码和三个答案发给我，我继续审查。